# Workshop 3W — Upsell Like an Engineer
Unit 3 — Programming for Intelligent Products

Today we’ll practise TDD (Red → Green → Refactor), experiment logging, and debugging discipline.

In [3]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
import mlflow, pandas as pd
print("MLflow:", mlflow.__version__)
print("Environment ready ✅")


Python: 3.14.2 (main, Sep 10 2026, 12:33:04) [GCC 13.3.0]
Platform: Linux-6.8.0-1064-azure-x86_64-with-glibc2.39
MLflow: 3.16.1
Environment ready ✅


In [4]:
def should_upsell(order: dict) -> bool:
    """Predict if a customer will buy a drink."""
    return order["loyalty_member"] == "yes"

    #return None  # start with a failing stub (RED)


# Positive test – loyalty at lunch → True
order_1a = {"time_of_day": "lunch", "loyalty_member": "yes"}
assert should_upsell(order_1a) is True, "FAIL: Loyalty at lunch should be upsold."
print("PASS: Test 1a")

# Negative test – non-loyalty at lunch → False (prevents `return True`)
order_1b = {"time_of_day": "lunch", "loyalty_member": "no"}
assert should_upsell(order_1b) is False, "FAIL: Non-loyalty at lunch should NOT be upsold."
print("PASS: Test 1b")


PASS: Test 1a
PASS: Test 1b


In [5]:
def should_upsell(order: dict) -> bool:
    """Predict if a customer will buy a drink."""
    return order["time_of_day"] == "lunch" and order["loyalty_member"] == "yes"

    #return None  # start with a failing stub (RED)


# Positive test – loyalty at lunch → True
order_1a = {"time_of_day": "lunch", "loyalty_member": "yes"}
assert should_upsell(order_1a) is True, "FAIL: Loyalty at lunch should be upsold."
print("PASS: Test 1a")

# Negative test – non-loyalty at lunch → False (prevents `return True`)
order_1b = {"time_of_day": "lunch", "loyalty_member": "no"}
assert should_upsell(order_1b) is False, "FAIL: Non-loyalty at lunch should NOT be upsold."
print("PASS: Test 1b")

# Evening loyalty → should NOT be upsold
order_2 = {"time_of_day": "evening", "loyalty_member": "yes"}
assert should_upsell(order_2) is False, "FAIL: Evening loyalty should NOT be upsold."
print("PASS: Test 2")


PASS: Test 1a
PASS: Test 1b
PASS: Test 2


In [6]:
try:
    import mlflow
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment("hotdog-upsell")
    with mlflow.start_run(run_name="Initial Loyalty Rule"):
        order = {"time_of_day": "lunch", "loyalty_member": "yes"}
        prediction = should_upsell(order)
        mlflow.log_params(order)
        mlflow.log_metric("prediction", int(bool(prediction)))
        mlflow.set_tag("rule_version", "v1")
    print("Experiment logged successfully ✅")
except Exception:
    print("[INFO] MLflow not installed — skipping logging safely.")

Experiment logged successfully ✅


In [ ]:
def should_upsell(order: dict) -> bool:
    """Predict if a customer will buy a drink."""
    if order.get("temperature") and  float(order.get("temperature")) >= 30:
        return True
    
    if order.get("order_size") and float(order.get("order_size")) >= 4:
        return True

    if order.get("time_of_day") and order.get("loyalty_member"):
        return order["time_of_day"] == "lunch" and order["loyalty_member"] == "yes"

    return None  # start with a failing stub (RED)


# Positive test – loyalty at lunch → True
order_1a = {"time_of_day": "lunch", "loyalty_member": "yes"}
assert should_upsell(order_1a) is True, "FAIL: Loyalty at lunch should be upsold."
print("PASS: Test 1a")

# Negative test – non-loyalty at lunch → False (prevents `return True`)
order_1b = {"time_of_day": "lunch", "loyalty_member": "no"}
assert should_upsell(order_1b) is False, "FAIL: Non-loyalty at lunch should NOT be upsold."
print("PASS: Test 1b")

# Evening loyalty → should NOT be upsold
order_2 = {"time_of_day": "evening", "loyalty_member": "yes"}
assert should_upsell(order_2) is False, "FAIL: Evening loyalty should NOT be upsold."
print("PASS: Test 2")

# Heatwave (temperature 30 or greater) → True
order_3 = {"time_of_day": "evening", "loyalty_member": "no", "temperature": 32}
assert should_upsell(order_3) is True, "FAIL: Hot weather should trigger upsell."
print("PASS: Test 3")

# Large order (order size 4 or greater) → True
order_4 = {"order_size": 4}
assert should_upsell(order_4) is True, "FAIL: Large orders should trigger upsell."
print("PASS: Test 4")

PASS: Test 1a
PASS: Test 1b
PASS: Test 2
PASS: Test 3
PASS: Test 4


In [26]:
def should_upsell(order: dict) -> bool:
    """Predict if a customer will buy a drink."""
    print(order)

    if order.get("order_size") and float(order.get("order_size")) >= 4:
            print("Large order (order size 4 or greater)")
            return True
    elif order.get("temperature") and  float(order.get("temperature")) >= 30:
        if order.get("time_of_day") == "lunch" and order.get("loyalty_member") == "no":
            print("Busy lunch heatwave (non-loyalty) should return False")
            return False
        else:
            print("Non-lunch heatwave should return True")
            return True
        
    return order.get("time_of_day") == "lunch" and order.get("loyalty_member") == "yes"

    #return None  # start with a failing stub (RED)


# Positive test – loyalty at lunch → True
print("Test 1a start")
order_1a = {"time_of_day": "lunch", "loyalty_member": "yes"}
assert should_upsell(order_1a) is True, "FAIL: Loyalty at lunch should be upsold."
print("PASS: Test 1a")

# Negative test – non-loyalty at lunch → False (prevents `return True`)
print("Test 1b start")
order_1b = {"time_of_day": "lunch", "loyalty_member": "no"}
assert should_upsell(order_1b) is False, "FAIL: Non-loyalty at lunch should NOT be upsold."
print("PASS: Test 1b")

# Evening loyalty → should NOT be upsold
print("Test 2 start")
order_2 = {"time_of_day": "evening", "loyalty_member": "yes"}
assert should_upsell(order_2) is False, "FAIL: Evening loyalty should NOT be upsold."
print("PASS: Test 2")

# Heatwave (temperature 30 or greater) → True
print("Test 3 start")
order_3 = {"time_of_day": "evening", "loyalty_member": "no", "temperature": 32}
assert should_upsell(order_3) is True, "FAIL: Hot weather should trigger upsell."
print("PASS: Test 3")

# Large order (order size 4 or greater) → True
print("Test 4 start")
order_4 = {"order_size": 4}
assert should_upsell(order_4) is True, "FAIL: Large orders should trigger upsell."
print("PASS: Test 4")

# Busy lunch heatwave (non-loyalty) → should NOT upsell
print("Test 5 start")
order_5 = {"time_of_day": "lunch", "loyalty_member": "no", "temperature": 32}
assert should_upsell(order_5) is False, "FAIL: Busy lunch heatwave (non-loyalty) should NOT upsell."
print("PASS: Test 5")

Test 1a start
{'time_of_day': 'lunch', 'loyalty_member': 'yes'}
PASS: Test 1a
Test 1b start
{'time_of_day': 'lunch', 'loyalty_member': 'no'}
PASS: Test 1b
Test 2 start
{'time_of_day': 'evening', 'loyalty_member': 'yes'}
PASS: Test 2
Test 3 start
{'time_of_day': 'evening', 'loyalty_member': 'no', 'temperature': 32}
Non-lunch heatwave should return True
PASS: Test 3
Test 4 start
{'order_size': 4}
Large order (order size 4 or greater)
PASS: Test 4
Test 5 start
{'time_of_day': 'lunch', 'loyalty_member': 'no', 'temperature': 32}
Busy lunch heatwave (non-loyalty) should return False
PASS: Test 5


In [27]:
def should_upsell(order: dict) -> bool:
    """Predict if a customer will buy a drink."""
    print(order)
    size = order.get("order_size",0)
    temperature = order.get("temperature",0)
    daytime = order.get("time_of_day","")
    loyaltyMember = order.get("loyalty_member","")

    if size >= 4:
            print("Large order (order size 4 or greater)")
            return True
    elif temperature >= 30:
        if daytime == "lunch" and loyaltyMember == "no":
            print("Busy lunch heatwave (non-loyalty) should return False")
            return False
        print("Non-lunch heatwave should return True")
        return True
        
    return daytime == "lunch" and loyaltyMember == "yes"

    #return None  # start with a failing stub (RED)


# Positive test – loyalty at lunch → True
print("Test 1a start")
order_1a = {"time_of_day": "lunch", "loyalty_member": "yes"}
assert should_upsell(order_1a) is True, "FAIL: Loyalty at lunch should be upsold."
print("PASS: Test 1a")

# Negative test – non-loyalty at lunch → False (prevents `return True`)
print("Test 1b start")
order_1b = {"time_of_day": "lunch", "loyalty_member": "no"}
assert should_upsell(order_1b) is False, "FAIL: Non-loyalty at lunch should NOT be upsold."
print("PASS: Test 1b")

# Evening loyalty → should NOT be upsold
print("Test 2 start")
order_2 = {"time_of_day": "evening", "loyalty_member": "yes"}
assert should_upsell(order_2) is False, "FAIL: Evening loyalty should NOT be upsold."
print("PASS: Test 2")

# Heatwave (temperature 30 or greater) → True
print("Test 3 start")
order_3 = {"time_of_day": "evening", "loyalty_member": "no", "temperature": 32}
assert should_upsell(order_3) is True, "FAIL: Hot weather should trigger upsell."
print("PASS: Test 3")

# Large order (order size 4 or greater) → True
print("Test 4 start")
order_4 = {"order_size": 4}
assert should_upsell(order_4) is True, "FAIL: Large orders should trigger upsell."
print("PASS: Test 4")

# Busy lunch heatwave (non-loyalty) → should NOT upsell
print("Test 5 start")
order_5 = {"time_of_day": "lunch", "loyalty_member": "no", "temperature": 32}
assert should_upsell(order_5) is False, "FAIL: Busy lunch heatwave (non-loyalty) should NOT upsell."
print("PASS: Test 5")

Test 1a start
{'time_of_day': 'lunch', 'loyalty_member': 'yes'}
PASS: Test 1a
Test 1b start
{'time_of_day': 'lunch', 'loyalty_member': 'no'}
PASS: Test 1b
Test 2 start
{'time_of_day': 'evening', 'loyalty_member': 'yes'}
PASS: Test 2
Test 3 start
{'time_of_day': 'evening', 'loyalty_member': 'no', 'temperature': 32}
Non-lunch heatwave should return True
PASS: Test 3
Test 4 start
{'order_size': 4}
Large order (order size 4 or greater)
PASS: Test 4
Test 5 start
{'time_of_day': 'lunch', 'loyalty_member': 'no', 'temperature': 32}
Busy lunch heatwave (non-loyalty) should return False
PASS: Test 5


In [28]:
print("### My Commit Log")
commits = [
    "feat: Add initial failing test for lunch loyalty",
    "fix: Implement lunch loyalty rule",
    "feat: Add evening loyalty test",
    "fix: Handle time-of-day logic",
    "refactor: Simplify function and add docstring",
    "fix: Add busy heatwave override",
]
for c in commits:
    print("-", c)


### My Commit Log
- feat: Add initial failing test for lunch loyalty
- fix: Implement lunch loyalty rule
- feat: Add evening loyalty test
- fix: Handle time-of-day logic
- refactor: Simplify function and add docstring
- fix: Add busy heatwave override
